# Application Example: Calculate the Depths of Offshore Wind Turbines from Raster and Vector Data Sources.

This example illustrates how to calculate the depths of offshore wind turbines in the North Sea. It showcases Geokit's capabilities in combining multiple geodata sources, whether raster or vector, to conduct analyses in a concise and simple way. First, the extent of the North Sea is determined using vector data stored in a shapefile (.shp). Then, the potential positions of offshore turbines in 2050 are located within the North Sea using a second shapefile (.shp). Finally, the depths at the turbine positions are extracted from bathymetry data, which is provided as raster data in GeoTIFF format (.tif). All intermediate results are visualised.

The Major steps are

1. Preparation
   1.1. Import Libraries
   1.2. Download Input Data 
2. Load, Extract and Display the Shape of the North Sea based on Vector data
3. Load and Display the Turbine Location
4. Select, Load and Warp Bathymetry Data
5. Extract Water depths at Offshore Turbine Locations  

## 1. Preparation

First all required libraries must be imported and the necessary input data must be downloaded

### 1.1 Import all required Libraries

In [1]:
### Import the necessary libraries
import geokit as gk
from os.path import join
from pooch import Unzip
import os
from pooch import DOIDownloader
import pathlib
import pooch

current_working_directory = pathlib.Path.cwd()
root_directory= current_working_directory.parent
data_directory= root_directory.joinpath("geokit","data")

c:\Users\j.belina\AppData\Local\miniforge3\envs\geokit_env\Lib\site-packages\osgeo\osr.py:410: FutureWarning: Neither osr.UseExceptions() nor osr.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


In [2]:


# The three datasources that are combined for this analyses are:



# 1. Download the data

# 1. Shapefile with sea basins -> filter which basin is in the North Sea; showcase the 'where' argument
# 2. Filter the GEBCO tiles that intersect with the North Sea basin -> gk.Extent.fromGeom(basins.geom).filterSources(path to the tiles)
# 3. Combine the GEBCO tiles to a single tile -> gk.algorithms.combineSimilarRasters
# 4. Warp northsea basin onto the raster to extract only the relevant part -> gk.RegionMask.warp
# 5. Shapefile with points of offshore wind points somewhere in the North Sea -> gk.vector.extractFeatures
# 6. Interpolate the depth of the points using the GEBCO tiles -> gk.raster.interpolateValues

### 1.2.1 Download Sea Basin Data

Sea Basin data is used to determine the shape and size of the North Sea. For this purpose, a shapefile from the 'IHO Sea Areas, version 3' is used, which can be found here:
https://doi.org/10.14284/323
The data is downloaded and extracted automatically in the next cell. The dataset is available under the CC BY-NC-SA 4.0 licence. You are not permitted to upload the data elsewhere.

In [ ]:

# Name of the archive
file_name_archive = r"iho.zip"

# Configure the extraction of the iho archive
# Extract the shapefiles into a folder called iho 
unpack_sea_basins = Unzip(extract_dir=data_directory.joinpath("iho"))

pooch.retrieve(
    url="https://geo.vliz.be/geoserver/wfs?request=getfeature&service=wfs&version=1.0.0&typename=MarineRegions:iho&outputformat=SHAPE-ZIP",
    known_hash="SHA256:1cedbad79d5b4b9dfae1a81350b699888d1bd469f6abb83fc290803062f675bc",
    processor=unpack_sea_basins,
    progressbar=True,
    path=data_directory,
    fname=file_name_archive,
)

file_name = r"iho.shp"
path_to_sea_basins = str(data_directory.joinpath("iho", file_name))
print(path_to_sea_basins)



0.00B [00:00, ?B/s]MB/s]
SHA256 hash of downloaded file: 1cedbad79d5b4b9dfae1a81350b699888d1bd469f6abb83fc290803062f675bc
Use this value as the 'known_hash' argument of 'pooch.retrieve' to ensure that the file hasn't changed if it is downloaded again in the future.
Unzipping contents of 'C:\Programming\geokit\geokit\data\iho.zip' to 'c:\Programming\geokit\geokit\data\iho'


c:\Programming\geokit\geokit\data\iho\iho.shp


### 1.2.2 Download Offsore Windturbine Location Dataset

Sea Basin data is used to determine the shape and size of the North Sea. For this purpose, a shapefile from the 'IHO Sea Areas, version 3' is used, which can be found here:
https://zenodo.org/records/10259046
The data is downloaded and extracted automatically in the next cell. The dataset is available under the CC BY-NC-SA 4.0 licence. You are not permitted to upload the data elsewhere.

In [ ]:
### Load 

# The second dataset is a vector dataset with plausible wind offshore locations in the north sea by 2050
# https://zenodo.org/records/10259046
turbine_location_handler = pooch.create(
    path=data_directory,
    # Use the figshare DOI
    # base_url="doi:10.5281/zenodo.14222865",
    base_url="doi:10.5281/zenodo.10259046",
    registry=None,
    version_dev=None,
)


# Automatically populate the registry
asd = turbine_location_handler.load_registry_from_doi()
print(asd)
# Fetch one of the files in the repository

unpack_zones_archive = Unzip(
    extract_dir=data_directory.joinpath("turbine_locations shapefile")
)
fname = turbine_location_handler.fetch(
    "turbine_locations shapefile.zip", processor=unpack_zones_archive
)



In [ ]:
# downloader = DOIDownloader()
# turbine_location_handler = pooch.create(
#     path=r"C:\Programming\geokit\Examples",
#     # Use the figshare DOI
#     # base_url="doi:10.5281/zenodo.14222865",
#     base_url="doi:10.5281/zenodo.17047388",
#     registry=None,
#     version_dev=None,

# )


# # Automatically populate the registry
# asd = turbine_location_handler.load_registry_from_doi()
# print(asd)
# # Fetch one of the files in the repository
# # Based on GEBCO
# # doi:10.5285/a29c5465-b138-234d-e053-6c86abc040b9
# unpack_zones_archive = Unzip(
#     extract_dir=current_working_directory.joinpath("GEBCO_bathymetry_tiles")
# )
# fname = turbine_location_handler.fetch(
#     "GEBCO_bathymetry_tiles.zip", processor=unpack_zones_archive,)


bathymetry_rasters_dir = join(data_directory, "GEBCO_bathymetry_tiles")

## 2. Load, Extract and Display the Region of the North Sea Based on Sea Basin Vector data

In [ ]:


# 1 Download the data
## Geokit provides functionality to automatically download the test data required
## Parts of the data is quite large so it might take a while to download



# # This is the path to the sea basin data
# sea_basins_path = join(data_dir, "vector", "World_Seas_IHO_v3", "World_Seas_IHO_v3.shp")
# url="https://doi.org/10.14284/323"

# url = "doi:10.6084/m9.figshare.14763051.v1/tiny-data.txt"
# Not using with Pooch.fetch so no need to pass an instance of Pooch
# url="doi.org/10.5281/zenodo.14222865/OSW_zones.png"

# pooch.retrieve("https://zenodo.org/records/10259046/OSW zones shapefile.zip",known_hash=None)






# # This is the path to the vector data with offshore wind points
# offshore_wind_points_path = join(data_dir, "vector", "turbine_locations shapefile", "turbine_locations.shp")


# # md5:7bfb4b091c4aa457b66d75fbca361b67
# # THis is 





In [ ]:
## Load and display Sea basin data

sea_basins = gk.vector.extractFeatures(
    source=path_to_sea_basins, where="NAME='North Sea'"
)

axh2 = gk.drawGeoms(sea_basins, srs=gk.srs.EPSG4326)

## 3. Load and Display the Turbine Location

In [ ]:
# file_name = r"World_Seas_IHO_v3.shp"
# path_to_sea_basins = str(
#     current_working_directory.joinpath("World_Seas_IHO_v3", file_name)
# )

# print(path_to_sea_basins)
# sea_basins = gk.vector.extractFeatures(
#     source=path_to_sea_basins, where="NAME='North Sea'"
# )

# axh2 = gk.drawGeoms(sea_basins, srs=gk.srs.EPSG4326)


In [ ]:

offshore_wind_points_path = current_working_directory.joinpath(
    "turbine_locations shapefile", "turbine_locations.shp"
)
# offshore_wind_points_path = current_working_directory.joinpath("osw_zones","turbine_locations.shp") 

offshore_wind_points = gk.vector.extractFeatures(source=str(offshore_wind_points_path))


In [ ]:
# load offshore wind points
# axh2 = gk.drawGeoms(offshore_wind_points, srs=gk.srs.EPSG4326, ax=axh1.ax)

axh1 = gk.drawGeoms(sea_basins, srs=gk.srs.EPSG4326)
axh2 = gk.drawGeoms(offshore_wind_points, srs=gk.srs.EPSG4326, ax=axh1.ax)

## 4. Load and Display Bathymetry Data

In [ ]:
## Check GEBCO tiles that overlap with the North Sea basin

bathymetry_datasets = list(gk.Extent.fromGeom(sea_basins.geom[0]).filterSources(bathymetry_rasters_dir + "/*.tif"))

In [ ]:
bathymetry_rasters_dir


In [ ]:
type(gk.Extent.fromGeom(sea_basins.geom[0]))

In [ ]:
bathymetry_datasets


In [ ]:
# combine the GEBCO tiles to a single raster
path_to_bathymetry_file=current_working_directory.joinpath("bathymetry_raster_NorthSea.tif")
path_to_bathymetry_file_str = str(path_to_bathymetry_file)
bathymetry_raster = gk.algorithms.combineSimilarRasters(
    bathymetry_datasets, output=path_to_bathymetry_file_str
)


In [ ]:
GEBCO_rasterInfo = gk.raster.rasterInfo(path_to_bathymetry_file_str)

assert GEBCO_rasterInfo.pixelHeight == GEBCO_rasterInfo.pixelWidth, (
    "Pixel height and width must be equal. Consider warping the raster to a square pixel size."
)

## create a region mask
northSeaMask = gk.RegionMask.fromGeom(
    sea_basins.geom[0], srs=gk.srs.EPSG4326, pixelRes=GEBCO_rasterInfo.pixelHeight
)


In [ ]:
## warp the region mask onto the GEBCO raster
northSeaRasterWarped = northSeaMask.warp(
    source=path_to_bathymetry_file_str,
    noData=GEBCO_rasterInfo.noData,
    returnMatrix=False,
)


In [ ]:
## draw the offshore wind points on the GEBCO raster
axh = gk.drawRaster(
    northSeaRasterWarped,
    srs=gk.srs.EPSG4326,
    cmap="YlGnBu",
    figsize=(6, 6),
    hideAxis=True,
    cbarTitle="Water Depth (m)",
    vmin=-600,
    vmax=200,
)

axh2 = gk.drawGeoms(sea_basins, srs=gk.srs.EPSG4326, ax=axh.ax, fc="none")
axh2 = gk.drawGeoms(offshore_wind_points, srs=gk.srs.EPSG4326, ax=axh.ax, markersize=1)


## 5. Extract Water depths at Offshore Turbine Locations  

In [ ]:
## retrieve the water depth at the offshore wind points
offshore_wind_points["depth"] = offshore_wind_points.geom.apply(
    lambda g: gk.raster.interpolateValues(
        source=northSeaRasterWarped, points=g, pointSRS=gk.srs.EPSG4326
    )
)


In [ ]:
offshore_wind_points.head()
